In [1]:
APPLICATION_NAME = "DEMO_DocGenerator"

In [2]:
import sys
from pathlib import Path

# Remonter les dossiers jusqu'à trouver lib/init/init_code.py
def find_app_home(sentinel: str ="lib/bootstrap/bootstrap.py"):
    current = Path.cwd().resolve()
    root = current.root
    while current != root:
        if (current / sentinel).is_file():
            return current
        current = current.parent
    raise FileNotFoundError(f"Impossible de trouver le fichier sentinelle : {sentinel}")

# Trouver et ajouter APPLICATION_HOME au sys.path
APPLICATION_HOME = find_app_home()
sys.path.insert(0, str(APPLICATION_HOME))
#print(f"APPLICATION_HOME set to: {APPLICATION_HOME}")

from lib.bootstrap.bootstrap import init_env
epy = init_env()


🔧  Application Context Initialized
──────────────────────────────────────────────────
🖥️  SYSTEM : Windows (Windows-10-10.0.26100-SP0)
🐍  PYTHON_VERSION : 3.11.9
🧪  INTERPRETER_PATH : c:\APPLICATIONS\python-3.11.9\python.exe
📦  ENVIRONMENT : ❌ Pas de virtualenv
📦  EPY_MODULES : appenv, cfgprops, cfgyaml, doc, load_class, log
📁  APPLICATION_HOME : C:\Users\bgonzale\Downloads\PYTHON_PROJECT\EssencePy-1
📛  APPLICATION_NAME : DEMO_DocGenerator
📘  ENV file loaded : C:\Users\bgonzale\Downloads\PYTHON_PROJECT\EssencePy-1\conf\env.conf
📘  PROPERTIES file loaded : C:\Users\bgonzale\Downloads\PYTHON_PROJECT\EssencePy-1\conf\application.properties
📘  YAML file loaded : C:\Users\bgonzale\Downloads\PYTHON_PROJECT\EssencePy-1/conf/application.yaml

✅  Context is ready.
──────────────────────────────────────────────────


## Documentation Generator

### listing fichiers

In [10]:
from pathlib import Path

def list_python_files(application_home):
    """
    Liste tous les fichiers *.py sous APPLICATION_HOME
    """
    application_home = Path(application_home).resolve()

    return sorted([
        p.resolve()
        for p in application_home.rglob("*.py")
        if p.is_file()
    ])


In [ ]:
files = list_python_files(APPLICATION_HOME)
for f in files:
    print(f)


### listings modules chargés et comparaison

In [12]:
import sys
from pathlib import Path

def list_loaded_project_modules(application_home):
    """
    Retourne les modules chargés par EPY
    appartenant au projet (APPLICATION_HOME)
    """
    application_home = Path(application_home).resolve()
    modules = {}

    for name, module in sys.modules.items():
        file = getattr(module, "__file__", None)
        if not file:
            continue

        try:
            file_path = Path(file).resolve()
        except Exception:
            continue

        if application_home in file_path.parents:
            modules[file_path] = {
                "module_name": name,
                "module": module
            }

    return modules


In [13]:
def compare_files_vs_loaded_modules(application_home):
    py_files = set(list_python_files(application_home))
    loaded_modules = list_loaded_project_modules(application_home)

    loaded_files = set(loaded_modules.keys())

    return {
        "all_python_files": py_files,
        "loaded_files": loaded_files,
        "not_loaded_files": py_files - loaded_files,
        "loaded_only": loaded_files - py_files,  # cas anormal
    }


In [ ]:
result = compare_files_vs_loaded_modules(APPLICATION_HOME)

print("=== FICHIERS PY NON CHARGÉS PAR EPY ===")
for f in sorted(result["not_loaded_files"]):
    print(f)

print("\n=== MODULES CHARGÉS ===")
for f in sorted(result["loaded_files"]):
    print(f)


### extract signature

In [15]:
import inspect

def function_signature(func):
    """
    Retourne la signature d'une fonction
    """
    try:
        return str(inspect.signature(func))
    except (ValueError, TypeError):
        return "()"


In [16]:
def extract_module_api(module):
    """
    Extrait fonctions, classes et méthodes
    d'un module déjà chargé
    """
    api = {
        "module": module.__name__,
        "file": getattr(module, "__file__", ""),
        "functions": [],
        "classes": []
    }

    for name, obj in inspect.getmembers(module):
        # Fonctions
        if inspect.isfunction(obj) and obj.__module__ == module.__name__:
            api["functions"].append({
                "name": name,
                "signature": function_signature(obj),
                "doc": inspect.getdoc(obj)
            })

        # Classes
        elif inspect.isclass(obj) and obj.__module__ == module.__name__:
            methods = []
            for m_name, m_obj in inspect.getmembers(obj, inspect.isfunction):
                if m_obj.__qualname__.startswith(obj.__name__):
                    methods.append({
                        "name": m_name,
                        "signature": function_signature(m_obj),
                        "doc": inspect.getdoc(m_obj)
                    })

            api["classes"].append({
                "name": name,
                "doc": inspect.getdoc(obj),
                "methods": methods
            })

    return api


In [17]:
def extract_all_loaded_apis(application_home):
    modules = list_loaded_project_modules(application_home)
    return [
        extract_module_api(info["module"])
        for info in modules.values()
    ]


In [ ]:
extraction = extract_all_loaded_apis(epy.APPLICATION_HOME)
for e in extraction:
    print(e)

## test final

In [3]:
from lib.bootstrap.docgenerator import DocGenerator
# from lib.bootstrap import APPLICATION_HOME

doc = DocGenerator(
    app_home=epy.APPLICATION_HOME,
    output_file=f"{epy.APPLICATION_HOME}/docs/technical.md",
    # user_module_paths=[
    #     f"{epy.APPLICATION_HOME}/lib/my_dummy_class.py",
    #     # f"{APPLICATION_HOME}/notebooks/scripts"
    # ]
)

doc.generate()

In [3]:
epy.doc.generate()